In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).cuda()

def kv(cache, layer):
    if hasattr(cache, "layers"):
        return cache.layers[layer].keys, cache.layers[layer].values
    if hasattr(cache, "key_cache"):
        return cache.key_cache[layer], cache.value_cache[layer]
    return cache[layer]

prompt = "The cat sat on the mat"
enc = tok(prompt, return_tensors="pt").to("cuda")
print("Tokens:", tok.convert_ids_to_tokens(enc.input_ids[0]))

# Prefill: the whole prompt in one pass
out = model(**enc, use_cache=True)
cache = out.past_key_values
k, v = kv(cache, 0)
print("\nAfter prefill, layer 0")
print("K shape:", tuple(k.shape), " = (batch, kv_heads, tokens, head_dim)")
print("V shape:", tuple(v.shape))
print("First token's key, first 5 numbers:", k[0, 0, 0, :5].tolist())

# Decode: one token at a time, reusing the cache
next_id = out.logits[:, -1].argmax(-1, keepdim=True)
for step in range(3):
    out = model(input_ids=next_id, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    k, v = kv(cache, 0)
    print(f"Decode step {step+1}: new token '{tok.decode(next_id[0])}', cache now {k.shape[2]} tokens")
    next_id = out.logits[:, -1].argmax(-1, keepdim=True)

layers = model.config.num_hidden_layers
total = 2 * layers * k.shape[1] * k.shape[2] * k.shape[3] * 2
print(f"\nTotal KV cache: {layers} layers x K and V = {total/1024:.1f} KB for {k.shape[2]} tokens")

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
ids = tok("What is a beautiful cat on this planet?").input_ids
print(list(zip(tok.convert_ids_to_tokens(ids), ids)))

In [ ]:
print(tok("What is a beautiful cat on this planet?").input_ids)

In [ ]:
enc = tok("The cat sat on the mat", return_tensors="pt").to("cuda")
out = model(**enc, use_cache=True)
k, v = kv(out.past_key_values, 0)
tokens = tok.convert_ids_to_tokens(enc.input_ids[0])

torch.set_printoptions(precision=2, sci_mode=False)
for i, t in enumerate(tokens):
    print(f"\nToken {i}: {t!r}")
    print("  Key  :", k[0, 0, i, :8].float().cpu())
    print("  Value:", v[0, 0, i, :8].float().cpu())

In [ ]:
print(enc.input_ids)
print(enc.input_ids.dtype)

In [ ]:
cat_id = enc.input_ids[0, 1]                     # the ID for " cat"
emb = model.get_input_embeddings()               # the big lookup table
vec = emb(cat_id)

print("Token:", tok.decode(cat_id))
print("ID:", cat_id.item())
print("Table size:", tuple(emb.weight.shape))    # (vocab, 896)
print("Vector length:", vec.shape[0])
print("First 8 numbers:", vec[:8].float().cpu())
print("First 8 numbers:", vec[:8].detach().float().cpu())

In [1]:
!pip install -q vllm

In [2]:
PREFIX = False   # then True

import time
from vllm import LLM, SamplingParams

doc = ("Invoice INV-2041 from Acme Supplies to Beltline Auto Body, Dallas TX. "
       "Line items: brake pads, rotors, labor, shop supplies, disposal fees. " * 40)
questions = [f"Question {i}: what is the total, and who is the vendor?" for i in range(64)]
prompts = [doc + "\n\n" + q for q in questions]
params = SamplingParams(max_tokens=50, temperature=0)

llm = LLM(model="Qwen/Qwen2.5-3B-Instruct-AWQ", dtype="half",
          gpu_memory_utilization=0.8, max_model_len=4096,
          enable_prefix_caching=PREFIX)
llm.generate(["warm up"], params)
t = time.time()
llm.generate(prompts, params)
print(f"prefix_caching={PREFIX}: {time.time()-t:.2f} s for 64 requests")

W1115 06:33:18.667000 159 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 06:33:18.669000 159 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1115 06:33:18.671000 159 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=159) INFO 11-15 06:33:33 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct-AWQ'}


config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

(MainProcess pid=159) INFO 11-15 06:33:52 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=159) INFO 11-15 06:33:52 [model.py:2170] Using max model len 4096
(MainProcess pid=159) INFO 11-15 06:33:55 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


(MainProcess pid=159) INFO 11-15 06:33:56 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (11 outputs left out)

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=264) WARNING 11-15 06:35:52 [jit_monitor.py:140] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 64/64 [01:06<00:00,  1.04s/it, est. speed input: 1436.78 toks/s, output: 47.99 toks/s]

prefix_caching=False: 66.95 s for 64 requests


In [ ]:
import os; os._exit(0)

In [1]:
PREFIX = True   # then True

import time
from vllm import LLM, SamplingParams

doc = ("Invoice INV-2041 from Acme Supplies to Beltline Auto Body, Dallas TX. "
       "Line items: brake pads, rotors, labor, shop supplies, disposal fees. " * 40)
questions = [f"Question {i}: what is the total, and who is the vendor?" for i in range(64)]
prompts = [doc + "\n\n" + q for q in questions]
params = SamplingParams(max_tokens=50, temperature=0)

llm = LLM(model="Qwen/Qwen2.5-3B-Instruct-AWQ", dtype="half",
          gpu_memory_utilization=0.8, max_model_len=4096,
          enable_prefix_caching=PREFIX)
llm.generate(["warm up"], params)
t = time.time()
llm.generate(prompts, params)
print(f"prefix_caching={PREFIX}: {time.time()-t:.2f} s for 64 requests")

W1115 06:49:09.538000 799 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 06:49:09.540000 799 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1115 06:49:09.542000 799 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=799) INFO 11-15 06:49:21 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct-AWQ'}
(MainProcess pid=799) INFO 11-15 06:49:21 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=799) INFO 11-15 06:49:21 [model.py:2170] Using max model len 4096
(MainProcess pid=799) INFO 11-15 06:49:23 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


(MainProcess pid=799) INFO 11-15 06:49:24 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(EngineCore pid=860) INFO 11-15 06:49:29 [core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='Qwen/Qwen2.5-3B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_awq, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=Observabi

... (9 outputs left out)

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=860) WARNING 11-15 06:50:39 [jit_monitor.py:140] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 64/64 [00:08<00:00,  7.70it/s, est. speed input: 11521.29 toks/s, output: 384.85 toks/s]

prefix_caching=True: 8.60 s for 64 requests
